# 00 · 환경 점검

실행 전 확인: `uv sync --group dev` 후 커널 **A-PI (uv 3.12)** 또는 `uv run jupyter lab` 으로 열었는지.

이 노트북은 **유료 호출을 하지 않는다**(health = 모델 목록 GET). 키 값은 어디에도 출력하지 않는다 — 존재 여부(bool)만.

순서: 00 → 10_openai → 20_anthropic → 30_gemini → 90_compare

In [ ]:
# 셀1. 버전 확인 — 시스템 Python 3.9 에서 열면 anthropic 0.x 가 깔려 있어 여기서 멈춘다(docs/HOWTO_RUN.md 문제 해결 참고)
import sys
import openai, anthropic
from google import genai

assert sys.version_info >= (3, 12), f'Python 3.12+ 필요: {sys.version}'
assert openai.__version__.startswith('3.'), f'openai 3.x 필요: {openai.__version__}'
assert anthropic.__version__.startswith('1.'), f'anthropic 1.x 필요: {anthropic.__version__}'
assert genai.__version__.startswith('2.'), f'google-genai 2.x 필요: {genai.__version__}'

print('python      ', sys.version.split()[0], '→', sys.executable)
print('openai      ', openai.__version__)
print('anthropic   ', anthropic.__version__)
print('google-genai', genai.__version__)

In [ ]:
# 셀2. 키 존재 여부 — 값은 절대 출력하지 않는다 (apicompare.config.keys() 는 bool 만 돌려준다)
import pandas as pd
from apicompare.config import keys

k = keys()
df_keys = pd.DataFrame({'env_var': list(k), 'set': list(k.values())})
df_keys['note'] = df_keys['env_var'].map({
    'OPENAI_API_KEY': '① ② ③ ⑥ ⑦',
    'OPENAI_ADMIN_KEY': '④ ⑤ Usage/Costs API (Owner 발급)',
    'ANTHROPIC_API_KEY': '① ② ③ ⑥ ⑦',
    'ANTHROPIC_ADMIN_KEY': '④ ⑤ usage_report/cost_report (조직 admin)',
    'GEMINI_API_KEY': '① ② ③ ⑥ ⑦ (④⑤ 는 API 없음)',
})
df_keys

In [ ]:
# 셀3. 세 Provider health() — 토큰 과금 없는 호출(GET models). 키 없으면 mode=NO_KEY, ok=False(AUTH) 가 정상.
import pandas as pd
from apicompare import PROVIDERS
from apicompare.providers import get_adapter

rows = []
for p in PROVIDERS:
    row = {'provider': p, 'mode': None, 'ok': None, 'status_code': None, 'latency_ms': None, 'error.category': None, 'note': ''}
    try:
        adapter = get_adapter(p, timeout_s=30, max_retries=0)
    except (ImportError, ModuleNotFoundError) as exc:
        row['note'] = f'어댑터 미작성 ({type(exc).__name__})'
        rows.append(row)
        continue
    row['mode'] = adapter.mode
    try:
        h = adapter.health()
        row.update(ok=h.ok, status_code=h.status_code, latency_ms=h.latency_ms,
                   **{'error.category': h.error.category if h.error else None}, note=h.checked_via)
    except Exception as exc:  # 어댑터는 예외를 던지지 않기로 했지만 방어
        row['note'] = f'health() 예외: {type(exc).__name__}'
    rows.append(row)

pd.DataFrame(rows)

In [ ]:
# 셀4. 단가표 — 3사 모두 단가를 API 로 제공하지 않아 pricing.PRICE_TABLE 을 로컬에서 유지한다(as_of·출처·인상 예고 포함)
import pandas as pd
from apicompare.pricing import PRICE_TABLE

df_price = pd.DataFrame([
    {
        'provider': p.provider, 'model': p.model, 'input $/1M': p.input, 'cached $/1M': p.cached_input,
        'cache_write_5m $/1M': p.cache_write_5m, 'output $/1M': p.output, 'as_of': p.as_of,
        'scheduled': '; '.join(f"{c['effective_from']}: ${c['input']}/${c['output']}" for c in p.scheduled_changes) or '',
        'note': p.note,
    }
    for p in PRICE_TABLE
])
pd.set_option('display.max_colwidth', 60)
df_price

In [ ]:
# 셀5. evidence 배지 3×7 — 아직 노트북 10/20/30 을 안 돌렸으면 전부 [BLOCKED: no evidence] 가 정상
import pandas as pd
from apicompare import PROVIDERS, PROVIDER_LABEL
from apicompare.evidence import summary

table = {}
for p in PROVIDERS:
    for row in summary(p):
        table.setdefault(row['item_ko'], {})[PROVIDER_LABEL[p]] = row['badge']
pd.DataFrame(table).T[[PROVIDER_LABEL[p] for p in PROVIDERS]]